In [69]:
import pandas as pd
import numpy as np
from sksurv.linear_model import CoxPHSurvivalAnalysis,CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest,ComponentwiseGradientBoostingSurvivalAnalysis
from MSB_package.msb_package import *
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
# from sksurv.util import Service
from sklearn.pipeline import Pipeline

# 1. Generate Synthetic Data with Predictive Signal
np.random.seed(432)
n_samples = 100

# Features
age = np.random.uniform(40, 80, n_samples)
bmi = np.random.uniform(20, 35, n_samples)
gene_a = np.random.normal(0, 1, n_samples)
gene_b = np.random.normal(0, 1, n_samples)

# Create a survival time influenced by Age and Gene_A
# risk = 0.05*age + 0.8*gene_a
risk_score = (0.05 * age) + (0.8 * gene_a)
time = np.exp(5 - 0.1 * risk_score) + np.random.normal(0, 2, n_samples)
time = np.maximum(time, 1)  # Ensure positive time
status = np.random.choice([True, False], n_samples, p=[0.8, 0.2])

X = pd.DataFrame({
    'age': age, 'bmi': bmi, 
    'gene_A': gene_a, 'gene_B': gene_b
})
X.index.name = 'SUBJID'

# Introduce Blockwise Missingness (30% of patients missing genomics)
X.iloc[70:, 2:] = np.nan 
X.iloc[:35, :2] = np.nan 
# Format target for sksurv
y = np.array([(s, t) for s, t in zip(status, time)],
             dtype=[('Status', 'bool'), ('Survival_in_days', 'float')])

# 2. Train/Test Split
# We use a standard split, but ensure the index remains intact for the MSB logic
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Define Modality Blocks
dict_block = pd.DataFrame([
    {'code': 'age', 'block': 'Clinical'},
    {'code': 'bmi', 'block': 'Clinical'},
    {'code': 'gene_A', 'block': 'Genomics'},
    {'code': 'gene_B', 'block': 'Genomics'}
])

# 4. Initialize and Fit MSB
# Using different base learners: Cox for Clinical, RSF for Genomics
final_est= Pipeline([('imp',KNNImputer().set_output(transform='pandas')),('meta',ComponentwiseGradientBoostingSurvivalAnalysis())])
# final_est = RandomSurvivalForest()

msb_model = MSB(
    estimators=[('cxgb', ComponentwiseGradientBoostingSurvivalAnalysis()),('coxnet',CoxnetSurvivalAnalysis(fit_baseline_model=True))],
    final_estimator = final_est,
    dict_block=dict_block,
    blocks=['Clinical', 'Genomics'],
    folds=3,
    id_name='SUBJID',
    impute=True 
)

msb_model.fit(X_train, y_train)

# 5. Evaluate
train_score = msb_model.score(X_train, y_train)
test_score = msb_model.score(X_test, y_test)

print(f"Train Concordance Index: {train_score:.3f}")
print(f"Test Concordance Index:  {test_score:.3f}")

# Predict survival function for the first two test patients
surv_funcs = msb_model.predict_survival_function(X_test.head(2))

Train Concordance Index: 0.806
Test Concordance Index:  0.761


In [70]:
cwxgb = Pipeline([('imp',KNNImputer().set_output(transform='pandas')),('meta',ComponentwiseGradientBoostingSurvivalAnalysis())])
cwxgb.fit(X_train, y_train)

# 5. Evaluate
train_score = cwxgb.score(X_train, y_train)
test_score = cwxgb.score(X_test, y_test)

print(f"Train Concordance Index: {train_score:.3f}")
print(f"Test Concordance Index:  {test_score:.3f}")


Train Concordance Index: 0.796
Test Concordance Index:  0.712
